In [ ]:
import numpy as np
import cv2
from sklearn.model_selection import train_test_split

# ==========================================
# KNN Classifier from Scratch for Natural Image Segmentation
# ==========================================

# 1. KNN Implementation
def knn_predict(Xtr, ytr, Xte, k):
    pred = []
    for x in Xte:
        d = np.sqrt(np.sum((Xtr - x)**2, axis=1))
        idx = np.argsort(d)[:k]
        votes = ytr[idx]
        pred.append(np.bincount(votes).argmax())
    return np.array(pred)

# 2. Main Execution Pipeline
def run_knn_segmentation(image_path, ref_path=None):
    print("Loading image and segmentations...")
    img = cv2.imread(image_path)
    if img is None:
        print("Image not found. Creating dummy data.")
        img = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
        ref = np.zeros((100, 100, 3), dtype=np.uint8) # Dummy reference
        ref[20:80, 20:80] = 255
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Load actual ref if available
        # ref = cv2.imread(ref_path)
        ref = np.zeros_like(img)
    
    h, w, _ = img.shape
    
    # 3. Extract RGB + Spatial Features
    print("Extracting features...")
    # Generate a mask for foreground (e.g. anything not fully white in ref)
    mask = ~np.all(ref == 255, axis=2)
    
    # If mask is empty, use all pixels
    if not np.any(mask): mask = np.ones((h, w), dtype=bool)
    
    rgb = img.reshape(-1, 3)[mask.ravel()] / 255.0
    yy, xx = np.mgrid[0:h, 0:w]
    xy = np.column_stack([xx.ravel()[mask.ravel()] / w,
                          yy.ravel()[mask.ravel()] / h])
    
    X = np.column_stack([rgb, 0.35 * xy])
    
    # Create dummy labels for this example (since unsupervised in original, or ref labels)
    y = np.random.randint(0, 5, len(X))
    
    # 4. Split data
    print("Splitting data into train/test...")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Reduce size for faster KNN computation (Optional, as KNN is O(N^2))
    if len(Xtr) > 2000:
        Xtr, ytr = Xtr[:2000], ytr[:2000]
        Xte, yte = Xte[:500], yte[:500]
    
    print(f"Training pixels: {len(Xtr)}")
    print(f"Testing pixels: {len(Xte)}")
    
    # 5. Prediction & Accuracy evaluation
    print("Evaluating K values...")
    ks = [1, 3, 5, 7, 9]
    accuracy = []
    
    for k in ks:
        yp = knn_predict(Xtr, ytr, Xte, k)
        acc = np.mean(yp == yte)
        accuracy.append(acc)
        print(f"K={k} -> Accuracy: {acc:.4f}")

    best_k = ks[np.argmax(accuracy)]
    print(f"Best K = {best_k}")
    print(f"Best Accuracy = {np.max(accuracy) * 100:.2f}%")
    
if __name__ == '__main__':
    run_knn_segmentation('sample.jpg')
